# Trajectory Analysis: Playback, Regions, and Movie Export

This tutorial walks through a trajectory analysis and visualization workflow using **only the core MolSysViewer API**:

1. **Load** a pre-packaged trajectory (alanine pentapeptide, 21 frames) from the demo catalog.
2. **Navigate** and play the trajectory frames programmatically.
3. **Annotate** structural subsets (backbone and sidechains) as independent **regions**.
4. **Color** the structure by mapping a custom list of scalar values (e.g. residue indices or fluctuations) to a colormap.
5. **Build and export** an animation combining a trajectory sweep, camera orbit, and visibility transitions using the movie builder.

**Requirements:** `molsysviewer` (and optionally `imageio` for exporting movies).

## 1. Imports

In [1]:
import molsysviewer as msv
from molsysviewer import demo

## 2. Load the Trajectory Demo

MolSysViewer includes a built-in demo catalog that ships with small, pre-compiled molecular systems (stored as `.h5msm` files). We load the `pentalanine` trajectory ensemble directly. Each access to `demo` returns a fresh, independent `MolSysView` instance.

In [2]:
view = demo["pentalanine"]
view.show()

## 3. Trajectory Playback and Navigation

The viewer supports multi-frame systems natively. We can check the number of structures, jump to a specific frame, or play/pause the animation.

In [3]:
# Set the active frame to frame 10 (0-indexed)
view.set_structure(10)

# Start playing the trajectory animation at 10 frames per second
view.play(fps=10)

# Pause the animation
view.pause()

## 4. Annotate Structural Regions

**Regions** represent structural subsets of the molecular system. They allow us to assign multiple simultaneous representations (e.g. cartoon for backbone, sticks for sidechains), toggle their visibility, or delete them independently.

Let's create two custom regions and hide the default global representation.

In [4]:
# Hide the default baseline representation of the whole system
view.whole.hide()

# Create a backbone region represented as a cartoon
backbone = view.new_region("backbone", tag="backbone", representation="cartoon")

# Create a sidechains region represented as sticks/licorice
sidechains = view.new_region("sidechain", tag="sidechains", representation="licorice")

# Focus the camera on the backbone region
view.focus_region("backbone")

## 5. Map Custom Values to Colors

We can map any list of scalar values (e.g., residue fluctuations, sequence index, or charge) directly onto the atoms of a region using a colormap. Here, we map 5 mock values (one per residue in the pentapeptide) onto the residues of our `backbone` region using the `coolwarm` palette.

In [5]:
# Mock values representing residue-level fluctuations (5 values for 5 residues)
residue_fluctuations = [0.12, 0.45, 0.88, 0.52, 0.21]

# Map these values to a colormap on the backbone region at the group (residue) level
backbone.set_color_by_values(
    values=residue_fluctuations,
    element="group",
    palette="coolwarm"
)

## 6. Build and Export a Movie

The `view.movie` timeline builder allows us to construct a sequence of keyframes, preview it in the notebook, and export it as an image sequence or a video file.

Let's define a timeline that sweeps through the trajectory frames while smoothly orbiting the camera.

In [6]:
# Clear any existing movie timeline keyframes
view.movie.clear()

# 1. Add a trajectory frame sweep from frame 0 to 20 over a duration of 4 seconds (4000 ms)
view.movie.add_structure_sweep(from_index=0, to_index=20, duration_ms=4000)

# 2. Add a camera orbit of 1 full turn around the structure over the same 4 seconds
view.movie.add_camera_orbit(duration_ms=4000, n_turns=1.0)

# Print the timeline summary info
print(view.movie.info())

### Preview and Export

We can preview the loop directly in our Jupyter notebook, or write it to a GIF/MP4 file.

In [7]:
# Play the movie loop in the browser canvas
view.movie.play(loop=True)

# Export the movie as an animated GIF (requires the imageio package)
# view.movie.export("pentalanine_trajectory.gif", fps=15)